The top-level formula for the Naive Bayes classifier is based on Bayes’ Theorem, which for a sample x and class c is:

$$
P(c\ |\ \mathbf{x}) \propto P(c) \cdot \prod_{i=1}^{n} P(x_i\ |\ c)
$$


P(c) is the prior probability of class c.
P(x_i | c) is the likelihood of feature x_i given class c, usually estimated by a Gaussian for continuous features[1][2][3].

The predicted class is:

$$
\hat{y} = \underset{c}{\arg\max}\ P(c)\ \prod_{i=1}^n P(x_i\ |\ c)
$$

In practical implementations, you use the logarithm to avoid numerical underflow:

$$
\log P(c\ |\ \mathbf{x}) = \log P(c) + \sum_{i=1}^{n} \log P(x_i\ |\ c)
$$

Choose the class with the largest log probability.[2][3]

[1](https://www.machinelearningmastery.com/naive-bayes-classifier-scratch-python/)
[2](https://www.geeksforgeeks.org/machine-learning/naive-bayes-scratch-implementation-using-python/)
[3](https://towardsdatascience.com/algorithms-from-scratch-naive-bayes-classifier-8006cc691493/)

In [1]:
import numpy as np

class NaiveBayes:
    def __init__(self):
        # Dictionary to hold prior probabilities for each class
        self.class_priors = {}
        # Dictionary to hold conditional probabilities/features likelihoods
        self.likelihoods = {}
        self.classes = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes, counts = np.unique(y, return_counts=True)
        
        # The goal of Naive Bayes is to assign, for sample x, the class y that maximizes P(y|x).
        # By Bayes Theorem:
        #     P(y|x) = P(x|y) * P(y) / P(x)
        # Since P(x) is the same for all classes, we ignore it for comparison.
        # We need to compute:
        #    - The prior P(y): probability of seeing class y (estimated by class_priors)
        #    - The likelihood P(x|y): probability of seeing sample x given class y (estimated by likelihoods)
        self.class_priors = {cls: count / n_samples for cls, count in zip(self.classes, counts)}
        self.likelihoods = {}
        # For Gaussian NB, we model likelihood P(x_i|y) as a Gaussian for each feature (mean/var per class)
        for cls in self.classes:
            X_cls = X[y == cls]
            self.likelihoods[cls] = {
                "mean": X_cls.mean(axis=0),
                "var": X_cls.var(axis=0) + 1e-9  # add epsilon to avoid div by zero
            }
    
    def gaussian(self,x,mean,var):
        exp = np.exp(-((x-mean)**2)/(2*var))
        const = 1 / np.sqrt(2*np.pi*var)
        return const * exp

    def predict(self, X):
        # Predict the class label for each sample in X.
        # For each class y, we compute the log posterior:
        #   log P(y|x) ∝ log P(y) + sum_i log P(x_i|y)
        n_samples, n_features = X.shape
        classifications = []

        for x in X:
            class_scores = {}
            for cls in self.classes:
                # get log(P(c)) -> prior
                log_prior = np.log(self.class_priors[cls])

                # get sum(log(P(x_i|c))) -> likelihood
                mean = self.likelihoods[cls]["mean"]
                var = self.likelihoods[cls]["var"]
                log_likelihood = np.sum(np.log(self.gaussian(x,mean,var)))

                # get log posterior, and assign it for this specific class
                log_posterior = log_prior + log_likelihood
                class_scores[cls] = log_posterior
            
            best_cls = max(class_scores, key=class_scores.get)
            classifications.append(best_cls)
    
        return np.array(classifications)

In [2]:
# Create a small toy dataset
# Two features, binary classification (0 or 1)
X_train = np.array([
    [1.0, 2.1],
    [2.0, 1.9],
    [0.9, 2.3],
    [3.1, 3.0],
    [2.8, 2.7],
    [3.0, 3.2]
])
y_train = np.array([0, 0, 0, 1, 1, 1])

X_test = np.array([
    [1.2, 2.0],   # Close to class 0
    [2.9, 3.1],   # Close to class 1
    [2.5, 2.5],   # Borderline
])
y_true = np.array([0, 1, 1])

# Instantiate and train
nb = NaiveBayes()
nb.fit(X_train, y_train)

# Do prediction
y_pred = nb.predict(X_test)

print("Test samples:")
print(X_test)
print("Predicted:", y_pred)
print("True:", y_true)
# Simple accuracy
acc = np.mean(y_pred == y_true)
print("Accuracy:", acc)

Test samples:
[[1.2 2. ]
 [2.9 3.1]
 [2.5 2.5]]
Predicted: [0 1 0]
True: [0 1 1]
Accuracy: 0.6666666666666666
